# 🎭 Detección de Puntos Clave Faciales con MediaPipe

**Materiales desarrollados por Matías Barreto, 2026**  
**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**  
* **Nomenclatura Oficial:** Procesamiento Digital de Imágenes  
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital  

---

## Objetivo

En este notebook vamos a detectar los **478 puntos clave** del rostro humano usando MediaPipe Face Mesh de Google. Vamos a trabajar sobre imágenes estáticas: cargar una foto, procesarla con el modelo y visualizar los landmarks dibujados sobre la imagen.

## Al terminar este material vamos a poder:

1. Explicar qué son los landmarks faciales y cuáles son sus aplicaciones reales.
2. Configurar y usar el detector Face Mesh de MediaPipe.
3. Convertir coordenadas normalizadas a posiciones en píxeles.
4. Visualizar los resultados de la detección sobre la imagen original.

## Microglosario

Antes de escribir código, definimos los términos que vamos a usar.

| Término | Definición | Analogía |
|---|---|---|
| **Landmark / Punto clave** | Coordenada (x, y, z) que representa un punto anatómico específico del rostro | Como los pines numerados en un mapa turístico: cada uno marca un lugar con nombre propio |
| **Face Mesh** | Red de 478 puntos distribuidos por todo el rostro que forman una malla geométrica | Como el alambrado de una escultura antes de aplicar el yeso: define la estructura tridimensional |
| **Coordenada normalizada** | Valor entre 0.0 y 1.0 que indica posición relativa independientemente del tamaño de la imagen | Como los porcentajes: 50% del ancho siempre es el centro, sin importar si la imagen mide 100 o 4000 píxeles |
| **BGR / RGB** | Dos formas de ordenar los canales de color de una imagen: Azul-Verde-Rojo (OpenCV) o Rojo-Verde-Azul (MediaPipe) | Como leer una lista de atrás para adelante: el contenido es el mismo, pero el orden cambia el significado |
| **`static_image_mode`** | Parámetro que le indica al modelo si va a procesar una imagen fija o cuadros de un video | Como avisarle al fotógrafo si va a sacar una foto de retrato o grabar un partido: cambia cómo prepara el equipo |

## ✦ Paso manual: de coordenada normalizada a píxel

Antes de ver el código, hagamos el cálculo a mano para entender qué hace el modelo con cada punto que detecta.

Supongamos que tenemos una imagen de **200 × 160 píxeles** y MediaPipe detecta la punta de la nariz en la posición normalizada `x = 0.50`, `y = 0.45`:

```
pixel_x = int(0.50 × 200) = int(100.0) = 100   ← mitad del ancho
pixel_y = int(0.45 × 160) = int(72.0)  =  72   ← un poco por encima del centro
```

Eso significa que el círculo verde se va a dibujar en la columna 100, fila 72.

El mismo cálculo se repite **478 veces** — una por cada landmark del Face Mesh. El código que vamos a ver más adelante hace exactamente eso en un bucle:

```python
coord_x_pixel = int(punto.x * ancho)
coord_y_pixel = int(punto.y * alto)
```

La normalización permite que el modelo funcione igual con una foto de celular de 720 píxeles y con una imagen de cámara profesional de 6000 píxeles.

## 1. Introducción: ¿Qué son los Landmarks Faciales?

### Una analogía simple

Imaginemos que queremos dibujar un retrato de alguien. ¿Qué puntos nos ayudarían a capturar mejor el rostro?

- Las esquinas de los ojos
- La punta de la nariz
- Las comisuras de los labios
- El contorno de la mandíbula
- Las cejas

Los **Landmarks Faciales** son exactamente eso: un conjunto de puntos específicos que "marcan" las características más importantes de un rostro.

### Definición técnica

Los landmarks faciales son **coordenadas (x, y, z)** de puntos clave predefinidos que representan características anatómicas de un rostro humano en una imagen digital.

Podemos pensarlos como un **mapa de ruta** que describe la geometría de una cara.

![MediaPipe Face Mesh](https://ai.google.dev/static/mediapipe/images/solutions/face_landmarker_keypoints.png?hl=es-419)

*Imagen: MediaPipe detecta 478 puntos en un rostro*

---

## 2. Aplicaciones en el Mundo Real

¿Por qué es importante detectar estos puntos? Acá hay algunos ejemplos cotidianos:

### Aplicaciones que usamos todos los días

1. **Filtros de Instagram y Snapchat**
   - Esos filtros que nos ponen orejas de perro o coronas de flores usan landmarks para saber exactamente dónde está la cara
   - Necesitan ubicar tus ojos, nariz y boca para colocar los efectos correctamente

2. **Face ID (desbloqueo facial)**
   - El celular analiza los landmarks del rostro para reconocer a su dueño
   - Compara la posición de los rasgos faciales con los datos guardados

3. **Videoconferencias con fondos virtuales**
   - Zoom, Teams y Meet detectan tu rostro para separarte del fondo
   - Usan landmarks para mantener tu cara siempre nítida mientras cambian el fondo

### Aplicaciones profesionales

| Área | Aplicación | Ejemplo |
|------|------------|----------|
| **Entretenimiento** | Animación facial para películas | Avatar, The Mandalorian |
| **Salud** | Detección de parálisis facial | Diagnóstico médico temprano |
| **Seguridad** | Reconocimiento facial | Aeropuertos, control de acceso |
| **Marketing** | Análisis de expresiones | Estudios de UX (experiencia de usuario) |
| **Gaming** | Control por gestos faciales | Juegos de realidad virtual |

---

## 3. Herramientas que Vamos a Usar

Para este ejercicio necesitamos dos bibliotecas principales:

### MediaPipe (Google)

**¿Qué es?** Una biblioteca de Google que nos da soluciones listas para usar en visión por computadora.

**¿Qué hace en este caso?** Detecta automáticamente 478 puntos en un rostro usando un modelo de inteligencia artificial ya entrenado.

**Analogía**: Es como tener un asistente experto que ya aprendió a reconocer rostros. No necesitamos entrenar nada, solo usarlo.

**Sitio oficial**: [MediaPipe Face Landmarker](https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker)

### OpenCV (cv2)

**¿Qué es?** La biblioteca estándar para procesamiento de imágenes en Python.

**¿Qué hace en este caso?**
- Carga imágenes desde archivos
- Convierte entre diferentes formatos de color
- Dibuja círculos y líneas sobre las imágenes
- Muestra las imágenes procesadas

**Analogía**: Es como Photoshop, pero para programadores.

---

## 4. Paso 1: Instalación de Bibliotecas

### ¿Qué vamos a hacer?

Vamos a instalar las bibliotecas necesarias para trabajar con imágenes y detectar rostros.

### Nota importante sobre el símbolo `!`

Cuando vemos `!` al principio de una línea en el notebook, significa que estamos ejecutando un **comando de terminal**, no código Python.

Es como abrir la consola de la computadora y escribir un comando.

### ¿Por qué dos instalaciones?

1. **Primera línea**: Actualizamos `numpy` (biblioteca de cálculos matemáticos)
2. **Segunda línea**: Instalamos `mediapipe` y `opencv-python`

### Advertencia

Es normal que aparezcan algunos mensajes en rojo (warnings). No hay que preocuparse; si al final dice "Successfully installed", todo está bien.

---

In [ ]:
# Instalamos las herramientas necesarias
# --quiet reduce la cantidad de texto que muestra la instalación
!pip install mediapipe opencv-python-headless numpy --quiet

# Verificamos que todo se importa correctamente
import mediapipe as mp
import cv2
import numpy as np

version_mediapipe = mp.__version__
version_opencv    = cv2.__version__
version_numpy     = np.__version__

print("✓ Entorno listo.")
print(f"  mediapipe  {version_mediapipe}")
print(f"  opencv     {version_opencv}")
print(f"  numpy      {version_numpy}")

## 5. Paso 2: Importar las Bibliotecas

### ¿Qué significa "importar"?

**Analogía**: Es como abrir las cajas de herramientas que acabamos de instalar.

Tener las bibliotecas instaladas no alcanza, necesitamos "activarlas" en nuestro código con el comando `import`.

### Las importaciones principales

1. **`cv2`**: El nombre corto de OpenCV (versión 2)
2. **`mediapipe as mp`**: MediaPipe, renombrado como `mp` para escribir menos
3. **`matplotlib.pyplot`**: Para visualizar imágenes directamente en el notebook

### ¿Por qué usamos alias (`as`)?

Escribir `mediapipe.solutions.face_mesh` es largo. Con el alias, podemos escribir solo `mp.solutions.face_mesh`.

Es como usar el apodo de alguien en lugar de su nombre completo cada vez.

---

In [ ]:
# Importamos las herramientas que vamos a usar en este notebook

import cv2          # OpenCV: para cargar y manipular imágenes
                    # cv2 = Computer Vision 2 (versión 2 de la biblioteca)

import mediapipe as mp  # MediaPipe: para detectar landmarks faciales
                        # 'as mp' nos permite escribir mp en lugar de mediapipe

import numpy as np  # NumPy: para operaciones con matrices numéricas

import matplotlib.pyplot as plt  # Para mostrar imágenes en el notebook

print("✓ Bibliotecas cargadas correctamente.")
print("  Estamos listos para empezar.")

## 6. Paso 3: Descargar una Imagen de Prueba

### ¿Por qué necesitamos una imagen?

Para detectar landmarks faciales, necesitamos una foto de un rostro. Podemos usar:
- La imagen que descargamos automáticamente
- Una foto propia
- Cualquier imagen de internet con un rostro visible

### Cómo usar una imagen propia (opcional)

1. Copiá tu foto en la misma carpeta donde está este notebook
2. Cambiá `ruta_imagen` en el código abajo por el nombre de tu archivo

### Recomendaciones para la imagen

- **Rostro visible**: Que se vea la cara completa (no de perfil)
- **Buena iluminación**: Evitá fotos muy oscuras
- **Un solo rostro**: Para este ejercicio, es mejor empezar con una sola persona
- **Tamaño moderado**: No hace falta que sea gigante (menos de 5MB está bien)

### ¿Qué hace este código?

1. Define la URL de una imagen de ejemplo
2. Intenta descargarla desde internet
3. La guarda con el nombre `rostro_ejemplo.jpg`
4. Si falla, te da instrucciones para subir tu propia foto

---

In [ ]:
# Descargamos una imagen de ejemplo de internet
# Podemos reemplazar la URL o cambiar ruta_imagen por una foto propia

import urllib.request

# URL de un retrato de ejemplo (Unsplash — licencia libre)
url_imagen  = "https://images.unsplash.com/photo-1500648767791-00dcc994a43e?q=80&w=1974&auto=format&fit=crop"

# Nombre con el que vamos a guardar el archivo localmente
ruta_imagen = "rostro_ejemplo.jpg"

texto_url = str(url_imagen)
print("Descargando imagen de ejemplo...")
print(f"  URL: {texto_url[:60]}...")

try:
    # urlretrieve descarga el archivo y lo guarda con el nombre indicado
    urllib.request.urlretrieve(url_imagen, ruta_imagen)
    print(f"✓ Imagen guardada como: {ruta_imagen}")

except Exception as error_descarga:
    # Si algo falla, mostramos el error y las alternativas
    texto_error = str(error_descarga)
    print(f"✗ Error al descargar: {texto_error}")
    print()
    print("Alternativa: copiá una imagen propia en esta carpeta")
    print(f"y cambiá ruta_imagen al nombre de ese archivo.")

## 7. Paso 4: Cargar y Preparar la Imagen

### ¿Qué vamos a hacer en este paso?

1. **Cargar la imagen** desde el archivo a la memoria de la computadora
2. **Verificar** que se cargó correctamente
3. **Convertir el formato de color** para que MediaPipe pueda procesarla

### Concepto importante: RGB vs BGR

Las computadoras representan colores mezclando Rojo, Verde y Azul (como un pintor mezcla pinturas).

**El problema**: Diferentes bibliotecas usan diferentes órdenes:
- **OpenCV usa BGR** (Azul, Verde, Rojo) - por razones históricas
- **MediaPipe usa RGB** (Rojo, Verde, Azul) - el orden más común

**Analogía**: Es como si un libro estuviera escrito de derecha a izquierda y otro de izquierda a derecha. El contenido es el mismo, pero hay que "traducir" el orden.

Por eso necesitamos convertir con `cv2.cvtColor()`.

### ¿Qué son las "dimensiones" de una imagen?

Una imagen digital es una cuadrícula de puntos (píxeles). Las dimensiones nos dicen:
- **Ancho**: cuántos píxeles tiene de izquierda a derecha
- **Alto**: cuántos píxeles tiene de arriba a abajo

Por ejemplo: 1920 × 1080 significa 1920 píxeles de ancho y 1080 de alto.

---

In [ ]:
# Cargamos la imagen desde el archivo y la preparamos para MediaPipe

print("=" * 50)
print("CARGANDO Y PREPARANDO LA IMAGEN")
print("=" * 50)
print()

# cv2.imread() lee el archivo y lo carga como una matriz de números
imagen = cv2.imread(ruta_imagen)

# Verificamos que la carga fue exitosa
# Si el archivo no existe, imread devuelve None
if imagen is None:
    print(f"✗ No se pudo cargar '{ruta_imagen}'")
    print("  Verificá que el archivo esté en la misma carpeta que este notebook.")

else:
    print("✓ Imagen cargada exitosamente.")
    print()

    # imagen.shape devuelve (alto, ancho, canales_de_color)
    # Accedemos a cada dimensión por índice para ser explícitos
    alto   = imagen.shape[0]   # cantidad de filas de píxeles
    ancho  = imagen.shape[1]   # cantidad de columnas de píxeles
    canales = imagen.shape[2]  # 3 canales: Azul, Verde, Rojo (formato BGR)

    print(f"  Ancho:   {ancho} píxeles")
    print(f"  Alto:    {alto} píxeles")
    print(f"  Canales: {canales} (formato BGR de OpenCV)")
    print()

    # Convertimos de BGR (OpenCV) a RGB (MediaPipe)
    # cv2.COLOR_BGR2RGB invierte el orden de los canales de color
    imagen_rgb = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)

    print("✓ Conversión de formato completada: BGR → RGB")
    print("  La imagen está lista para MediaPipe.")

## 8. Paso 5: Configurar el Detector de MediaPipe

### ¿Qué vamos a hacer?

Vamos a crear y configurar el "detector" de MediaPipe. Pensemos en esto como configurar una cámara antes de tomar una foto:
- ¿Qué tan sensible debe ser?
- ¿Cuántos rostros debe buscar?
- ¿Qué nivel de precisión queremos?

### Explicación de los parámetros

MediaPipe nos permite ajustar varios parámetros. Acá está cada uno explicado en detalle:

#### 1. `static_image_mode=True`

**¿Qué significa?**  
Le decimos a MediaPipe que vamos a procesar **una imagen fija** (no un video).

**¿Por qué es importante?**  
- Si fuera `False`, MediaPipe optimizaría para video (asume que los frames son secuenciales)
- Con `True`, cada imagen se procesa independientemente con máxima precisión

**Analogía**: Es como decirle a un fotógrafo "tomá una foto perfecta" vs "filmá un video continuo".

#### 2. `max_num_faces=1`

**¿Qué significa?**  
Buscá como máximo **1 rostro** en la imagen.

**¿Por qué lo ponemos en 1?**  
- Para este ejemplo, queremos simplificar y trabajar con un solo rostro
- Podés cambiarlo a 2, 3, o más si tu imagen tiene varias personas

**Nota**: Más rostros = más tiempo de procesamiento

#### 3. `refine_landmarks=True`

**¿Qué significa?**  
Activá el refinamiento de landmarks (puntos más precisos, especialmente en ojos e iris).

**¿Qué hace exactamente?**  
- Con `False`: detecta ~468 puntos básicos
- Con `True`: detecta 478 puntos (agrega puntos extra en los iris)

**Cuándo usar cada opción:**
- `True`: Si necesitás precisión en los ojos (filtros, seguimiento de mirada)
- `False`: Si solo necesitás el contorno general del rostro

#### 4. `min_detection_confidence=0.5`

**¿Qué significa?**  
El nivel mínimo de "confianza" para considerar que encontró un rostro.

**Escala**: va de 0.0 a 1.0 (0% a 100%)

**Ejemplos:**
- `0.5` (50%): Nivel medio - balancea precisión y flexibilidad
- `0.8` (80%): Muy estricto - solo rostros muy claros
- `0.3` (30%): Más permisivo - detecta rostros incluso en condiciones difíciles

**¿Cuándo ajustarlo?**
- Bajá el valor si tus fotos son oscuras o borrosas
- Subí el valor si querés evitar "falsos positivos" (cosas que no son rostros)

### ¿Por qué cerramos el detector al final?

**Analogía**: Es como cerrar un programa que ya no usás para liberar memoria RAM.

MediaPipe carga un modelo de inteligencia artificial que ocupa memoria. Cuando terminamos, es buena práctica liberarlo con `.close()`.

---

In [ ]:
# Configuramos y ejecutamos el detector de Face Mesh

print("=" * 50)
print("CONFIGURANDO EL DETECTOR DE LANDMARKS")
print("=" * 50)
print()

# Accedemos al módulo Face Mesh de MediaPipe
modulo_face_mesh = mp.solutions.face_mesh

# Creamos el detector con los parámetros que necesitamos
# static_image_mode=True: procesamos una imagen fija, no un video
# max_num_faces=1: buscamos un solo rostro en la imagen
# refine_landmarks=True: activa 478 puntos (vs 468 sin refinamiento)
# min_detection_confidence=0.5: confianza mínima del 50% para aceptar una detección
detector_facial = modulo_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5
)

print("✓ Detector inicializado.")
print("  Modo:             imagen estática")
print("  Rostros máximos:  1")
print("  Refinamiento:     activado (478 landmarks)")
print("  Confianza mínima: 50%")
print()

# Procesamos la imagen — este es el paso donde el modelo hace su trabajo
# .process() recibe la imagen en RGB y devuelve todos los rostros detectados
print("Procesando imagen...")
resultados = detector_facial.process(imagen_rgb)

# Cerramos el detector para liberar memoria
detector_facial.close()

print("✓ Procesamiento completado.")
print("  Los resultados están guardados en la variable 'resultados'.")

## 9. Paso 6: Dibujar los Landmarks sobre la Imagen

### ¿Qué vamos a hacer?

Ahora viene la parte visual: vamos a dibujar círculos verdes en cada uno de los 478 puntos detectados.

### Conceptos importantes

#### 1. Coordenadas normalizadas vs coordenadas de píxeles

MediaPipe nos da coordenadas **normalizadas**: valores entre 0 y 1.

**¿Por qué normalizar?**  
Para que funcione con cualquier tamaño de imagen.

**Ejemplo:**
- Coordenada normalizada: `x = 0.5` (significa "en la mitad horizontalmente")
- Si la imagen tiene 1000 píxeles de ancho: `0.5 × 1000 = 500` píxeles
- Si la imagen tiene 2000 píxeles de ancho: `0.5 × 2000 = 1000` píxeles

**Fórmula de conversión:**
```
coord_x_pixel = coord_x_normalizada × ancho_imagen
coord_y_pixel = coord_y_normalizada × alto_imagen
```

#### 2. Sistema de coordenadas

```
(0,0) -----------> X (ancho)
  |
  |
  |
  v
  Y (alto)
```

- El punto (0,0) está en la **esquina superior izquierda**
- X aumenta hacia la derecha
- Y aumenta hacia abajo (¡al revés de matemática!)

#### 3. La función `cv2.circle()`

Esta función dibuja un círculo en una imagen.

**Sintaxis:**
```python
cv2.circle(
    imagen,          # Dónde dibujar
    (x, y),          # Centro del círculo (tupla con coordenadas)
    radio,           # Tamaño del círculo en píxeles
    (B, G, R),       # Color en formato BGR
    grosor           # Grosor del borde (-1 = relleno completo)
)
```

**Nuestros valores:**
- Radio: `2` píxeles (círculos pequeños)
- Color: `(0, 255, 0)` = verde en BGR
- Grosor: `-1` = círculo completamente relleno

#### 4. ¿Por qué hacemos una copia de la imagen?

```python
imagen_con_puntos = imagen.copy()
```

Si modificamos `imagen` directamente, perdemos la original. Con `.copy()` creamos una versión duplicada que podemos modificar sin afectar el original.

**Analogía**: Es como hacer una fotocopia para escribir sobre ella, manteniendo el original intacto.

### Flujo del código

```
1. Crear copia de la imagen
   ↓
2. ¿Se detectaron rostros?
   ↓ SÍ
3. Tomar el primer rostro
   ↓
4. Para cada uno de los 478 puntos:
   → Convertir coordenadas normalizadas a píxeles
   → Dibujar círculo verde
   ↓
5. Mostrar imagen resultante
```

---

In [ ]:
# Dibujamos los 478 landmarks sobre la imagen

print("=" * 50)
print("DIBUJANDO LANDMARKS EN LA IMAGEN")
print("=" * 50)
print()

# Trabajamos sobre una copia para no modificar la imagen original
imagen_con_puntos = imagen.copy()

# Extraemos las dimensiones por índice (sin usar _ para el tercer valor)
alto    = imagen.shape[0]   # filas de píxeles
ancho   = imagen.shape[1]   # columnas de píxeles

print(f"Dimensiones de trabajo: {ancho} × {alto} píxeles")
print()

# Verificamos si MediaPipe detectó algún rostro
if resultados.multi_face_landmarks:

    print("✓ Rostro detectado.")
    print()

    # Tomamos el primer (y único) rostro de la lista
    rostro = resultados.multi_face_landmarks[0]

    # Convertimos los landmarks a una lista para recorrerla con índices
    lista_landmarks = list(rostro.landmark)
    cantidad_landmarks = len(lista_landmarks)

    print(f"  Landmarks detectados: {cantidad_landmarks}")
    print("  Dibujando puntos...")
    print()

    # Recorremos cada landmark y lo dibujamos como un círculo verde
    for i in range(len(lista_landmarks)):
        punto = lista_landmarks[i]

        # Cada punto tiene coordenadas normalizadas (entre 0.0 y 1.0)
        # Las convertimos a píxeles multiplicando por el tamaño de la imagen
        coord_x_pixel = int(punto.x * ancho)
        coord_y_pixel = int(punto.y * alto)

        # Dibujamos un círculo verde en esa posición
        cv2.circle(
            imagen_con_puntos,               # imagen donde dibujamos
            (coord_x_pixel, coord_y_pixel),  # centro del círculo
            2,                               # radio: 2 píxeles
            (0, 255, 0),                     # color: verde en formato BGR
            -1                               # -1 = círculo relleno
        )

    print(f"✓ {cantidad_landmarks} puntos dibujados sobre la imagen.")
    print()

    # Mostramos la imagen resultante con matplotlib
    # Necesitamos convertir de BGR (OpenCV) a RGB (matplotlib)
    imagen_para_mostrar = cv2.cvtColor(imagen_con_puntos, cv2.COLOR_BGR2RGB)

    figura, eje = plt.subplots(1, 2, figsize=(14, 6))

    eje[0].imshow(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
    eje[0].set_title("Imagen original")
    eje[0].axis('off')

    eje[1].imshow(imagen_para_mostrar)
    eje[1].set_title(f"Face Mesh — {cantidad_landmarks} landmarks")
    eje[1].axis('off')

    plt.tight_layout()
    plt.show()

    print("Leyenda:")
    print("  Cada punto verde es un landmark facial.")
    print("  Los 478 puntos están distribuidos por ojos, nariz, boca, frente y mandíbula.")

else:
    print("✗ No se detectó ningún rostro en la imagen.")
    print()
    print("Posibles causas:")
    print("  - El rostro está de perfil o muy inclinado")
    print("  - La imagen está muy oscura o borrosa")
    print("  - El rostro es muy pequeño dentro de la imagen")
    print()
    print("Sugerencias:")
    print("  - Probá con otra imagen de frente y buena iluminación")
    print("  - Bajá min_detection_confidence a 0.3 en la celda anterior")

## 10. Reflexión y Conclusiones

### ¿Qué acabamos de hacer?

En este notebook completamos un flujo completo de visión por computadora:

1. **Instalación de herramientas** (MediaPipe, OpenCV)
2. **Carga de imagen** desde internet o archivo local
3. **Preprocesamiento** (conversión de formato de color)
4. **Detección con IA** (modelo pre-entrenado de MediaPipe)
5. **Visualización de resultados** (dibujado de landmarks)

### Lo que aprendimos

**Conceptos técnicos:**
- Qué son los landmarks faciales y sus aplicaciones
- Diferencia entre coordenadas normalizadas y píxeles
- Formatos de color (RGB vs BGR)
- Uso de modelos pre-entrenados

**Habilidades prácticas:**
- Usar bibliotecas de Python (importar y configurar)
- Trabajar con imágenes en código
- Interpretar y visualizar resultados de IA
- Debugging básico (qué hacer si algo falla)

### Próximos pasos

Ahora que tenés los landmarks, podés:

**Nivel básico:**
- Cambiar el color o tamaño de los puntos
- Probar con diferentes fotos (tuyas, de familiares)
- Detectar múltiples rostros (cambiar `max_num_faces`)

**Nivel intermedio:**
- Medir distancias entre puntos (ej: ancho de la boca)
- Dibujar líneas conectando landmarks (crear una "malla")
- Detectar expresiones básicas (boca abierta/cerrada)

**Nivel avanzado:**
- Crear filtros tipo Snapchat (superponer imágenes)
- Analizar expresiones faciales automáticamente
- Trabajar con video en tiempo real
- Crear una aplicación web (Streamlit/Gradio)

### Recursos adicionales

**Documentación oficial:**
- [MediaPipe Face Landmarker](https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker)
- [OpenCV Python Tutorials](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html)

**Tutoriales en video:**
- Buscá en YouTube: "MediaPipe Face Mesh tutorial"
- Canal recomendado: "Nicholas Renotte" (en inglés, con subtítulos)

### Desafío final

**Probá modificar el código para:**
1. Dibujar solo los landmarks de los ojos (landmarks 33, 133, 362, 263)
2. Usar diferentes colores para diferentes partes del rostro
3. Agregar texto con el número de cada landmark

**Pista**: Vas a necesitar usar condiciones (`if`) y la función `cv2.putText()` para el texto.

---

**¡Felicitaciones por completar este tutorial!**

Ahora entendés cómo funcionan tecnologías que usás todos los días (filtros de redes sociales, reconocimiento facial, etc.).

Este es el primer paso en el fascinante mundo de la visión por computadora.

---

## 11. Actividades Extras (Opcional)

### Actividad 1: Experimentar con los parámetros

Volvamos a la celda donde configuramos el detector y probemos cambiar:

```python
# Opción 1: Detectar hasta 3 rostros
max_num_faces=3

# Opción 2: Ser más permisivo con la detección
min_detection_confidence=0.3

# Opción 3: Desactivar el refinamiento
refine_landmarks=False  # Detectará 468 puntos en vez de 478
```

**✎ Para pensar:**  
¿Cómo cambian los resultados con cada configuración?

---

### Actividad 2: Analizar las coordenadas

Agregamos este código en una nueva celda para ver las coordenadas del primer landmark:

```python
# Ver las coordenadas del primer punto
if resultados.multi_face_landmarks:
    primer_punto = resultados.multi_face_landmarks[0].landmark[0]
    print(f"Coordenadas del primer landmark:")
    print(f"  x (normalizada): {primer_punto.x:.4f}")
    print(f"  y (normalizada): {primer_punto.y:.4f}")
    print(f"  z (profundidad):  {primer_punto.z:.4f}")
    print()
    print(f"  x (píxeles): {int(primer_punto.x * ancho)}")
    print(f"  y (píxeles): {int(primer_punto.y * alto)}")
```

**Pregunta para reflexionar:**  
¿Por qué las coordenadas normalizadas son útiles?

---

### Actividad 3: Contar landmarks por región

Investigá en la [documentación de MediaPipe](https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker) cuáles son los rangos de índices para:
- Ojos
- Boca
- Contorno facial

Intentá crear un código que cuente cuántos landmarks hay en cada región.

---

### Actividad 4: Medir distancias

Usando el **Teorema de Pitágoras**, calculá la distancia entre dos puntos:

```python
import math

# Ejemplo: distancia entre los puntos 33 y 133 (esquinas externas de los ojos)
if resultados.multi_face_landmarks:
    landmarks = resultados.multi_face_landmarks[0].landmark
    
    punto1 = landmarks[33]
    punto2 = landmarks[133]
    
    # Convertir a píxeles
    x1 = punto1.x * ancho
    y1 = punto1.y * alto
    x2 = punto2.x * ancho
    y2 = punto2.y * alto
    
    # Teorema de Pitágoras: distancia = √((x2-x1)² + (y2-y1)²)
    distancia = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    
    print(f"Distancia entre esquinas de los ojos: {distancia:.2f} píxeles")
```

**Desafío:**  
Calculá el ancho de la boca (distancia entre las comisuras de los labios).

---

## Referencias y Créditos

### Documentación oficial

- [MediaPipe Face Landmarker](https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker?hl=es-419)
- [OpenCV Python Tutorials](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html)
- [Google Colab](https://colab.research.google.com/)

### Bibliotecas utilizadas

- **MediaPipe** - Google LLC (Apache License 2.0)
- **OpenCV** - OpenCV Team (Apache License 2.0)
- **NumPy** - NumPy Developers (BSD License)

### Imágenes de ejemplo

- Imagen de rostro: [Unsplash](https://unsplash.com/) (Licencia libre)

---

**Materiales desarrollados por Matías Barreto, 2026**  
**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**

---